<a href="https://colab.research.google.com/github/xxKeyaxx/Applying-Transfer-Learning-on-Various-GNN-Model-Training-in-Indoor-Positioning-System-Tasks/blob/main/(New)_Indoor_Positioning_System_Skripsi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import pandas as pd
import numpy as np
from torch_geometric.data import Data, Batch
from torch_geometric.nn import SAGEConv
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

test_file_path = '/content/drive/My Drive/UTSIndoorLoc/UTS_test.csv'
train_file_path = '/content/drive/My Drive/UTSIndoorLoc/UTS_train.csv'
val_file_path = '/content/drive/My Drive/UTSIndoorLoc/UTS_val.csv'
edge_file_path = '/content/drive/My Drive/UTSIndoorLoc/UTS_edge.csv'

df_test = pd.read_csv(test_file_path)
print(df_test.shape[0])
# df_test.drop_duplicates(inplace=True)

df_train = pd.read_csv(train_file_path)
print(df_train.shape[0])
# df_train.drop_duplicates(inplace=True)
print(df_train.head())  # Preview the data

df_val = pd.read_csv(val_file_path)
print(df_val.shape[0])
# df_val.drop_duplicates(inplace=True)
print(df_val.head())  # Preview the data

df_edge = pd.read_csv(edge_file_path)
print(df_edge.shape[0])
# df_edge.drop_duplicates(inplace=True)
print(df_edge.head())  # Preview the data

Mounted at /content/drive
388
6109
   index  WAP001  WAP002  WAP003  WAP004  WAP005  WAP006  WAP007  WAP008  \
0   5764     100     100     100     100     100     100     100     100   
1   1340     100     100     100     100     100     100     100     100   
2   1941     -86     -88     -87     100     100     100     100     100   
3    718     100     100     100     100     100     100     100     100   
4   1577     -85     -85     -85     100     100     100     100     100   

   WAP009  ...  WAP588  WAP589    Pos_x     Pos_y  Floor_ID  Building_ID  \
0     100  ...     100     100  21.0844  20.90850         9       New IT   
1     100  ...     100     100  37.6851   9.91089         6       New IT   
2     100  ...     100     100  64.3976  10.83900         4       New IT   
3     100  ...     100     100  42.0692  18.94720         7       New IT   
4     100  ...     100     100  62.3207  18.95800         5       New IT   

   User_ID  Phone_type            Time  is_edge  
0

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Initialize Min-Max Scaler
scaler = MinMaxScaler()

def preprocess_data(data):
    # Replace 100 with 0 (no detection)
    wap_cols = [col for col in data.columns if col.startswith('WAP')]
    sensor_cols = wap_cols + ['Floor_ID']
    # sensor_cols = wap_cols
    # data[wap_cols] = data[wap_cols].replace(100, -97)

    # normalize floor_ID
    # columns_to_normalize = ['Floor_ID']
    # columns_to_normalize = sensor_cols
    # data[columns_to_normalize] = scaler.fit_transform(data[columns_to_normalize])

    # data[sensor_cols] = data[sensor_cols].replace(100, 0)
    return data, sensor_cols

df_train, sensor_cols = preprocess_data(df_train)
# df_train = df_train[df_train['Floor_ID'] == 12]

# Load and preprocess the new dataset
df_test, sensor_cols = preprocess_data(df_test)
df_val, sensor_cols = preprocess_data(df_val)
df_edge, sensor_cols = preprocess_data(df_edge)
df_train = pd.concat([df_train, df_edge], ignore_index=True)
# max_floor = 12
# df_test = df_test[df_test['Floor_ID'] == max_floor]

GAT

In [ ]:
# # Create graph data for each floor
# def create_graphs_by_floor(data, sensor_cols):
#     graphs = {}
#     for floor_id in data['Floor_ID'].unique():
#         floor_data = data[data['Floor_ID'] == floor_id]
#         x = torch.tensor(floor_data[sensor_cols].values, dtype=torch.float)
#         y = torch.tensor(floor_data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

#         # Create dummy edges (fully connected graph for simplicity)
#         edge_index = torch.combinations(torch.arange(x.size(0)), r=2).T
#         edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

#         graphs[floor_id] = Data(x=x, edge_index=edge_index, y=y)
#     return graphs

# graphs_by_floor = create_graphs_by_floor(df_train, sensor_cols)

from sklearn.metrics.pairwise import euclidean_distances
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import pairwise_distances
import numpy as np

# # Set seeds for reproducibility
# torch.manual_seed(42)
# np.random.seed(42)

# # If using GPU
# if torch.cuda.is_available():
#     torch.cuda.manual_seed(42)
#     torch.cuda.manual_seed_all(42)  # For multi-GPU setups
#     torch.backends.cudnn.deterministic = True
#     torch.backends.cudnn.benchmark = False
#     torch.use_deterministic_algorithms(True)

# def create_knn_graphs_by_floor(data, sensor_cols, k=5):
#     graphs = {}
#     for floor_id in data['Floor_ID'].unique():
#         floor_data = data[data['Floor_ID'] == floor_id]
#         x = torch.tensor(floor_data[sensor_cols].values, dtype=torch.float)
#         y = torch.tensor(floor_data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

#         # Compute pairwise distances
#         distances = euclidean_distances(floor_data[sensor_cols])
#         edge_index = []

#         # For each node, find k nearest neighbors
#         for i in range(distances.shape[0]):
#             neighbors = np.argsort(distances[i])[:k]  # Get indices of k nearest neighbors
#             for j in neighbors:
#                 edge_index.append([i, j])

#         # Convert edge list to PyTorch Geometric format
#         edge_index = torch.tensor(edge_index, dtype=torch.long).T

#         graphs[floor_id] = Data(x=x, edge_index=edge_index, y=y)
#     return graphs

import copy

def create_knn_graphs_by_floor(data, sensor_cols, k):
    graphs = {}
    edge_copy = copy.deepcopy(df_edge)
    for floor_id in data['Floor_ID'].unique():
        # Filter data for the current floor
        floor_data = data[data['Floor_ID'] == floor_id]
        x = torch.tensor(floor_data[sensor_cols].values, dtype=torch.float)
        y = torch.tensor(floor_data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

        edge_data = edge_copy[edge_copy['Floor_ID'] == floor_id]
        # # Compute pairwise cosine similarities
        # similarities = cosine_similarity(floor_data[sensor_cols])
        # edge_index = []

        # # For each node, find k most similar neighbors
        # for i in range(similarities.shape[0]):
        #     # Get indices of k most similar neighbors (excluding itself)
        #     neighbors = np.argsort(similarities[i])[-(k + 1):-1][::-1]  # Exclude self and take top k
        #     for j in neighbors:
        #         edge_index.append([i, j])

        # # Convert edge list to PyTorch Geometric format
        # edge_index = torch.tensor(edge_index, dtype=torch.long).T

        # # Store the graph for the current floor
        # graphs[floor_id] = Data(x=x, edge_index=edge_index, y=y)

        # Compute pairwise cosine similarities between floor data and X_edge
        similarities = cosine_similarity(floor_data[sensor_cols].values, edge_data[sensor_cols].values)
        edge_index = []

        # For each node in the current floor, find k most similar neighbors in X_edge
        for i in range(similarities.shape[0]):
            # Get indices of k most similar neighbors in X_edge
            neighbors = np.argsort(similarities[i])[-k:][::-1]  # Take top k
            for j in neighbors:
                edge_index.append([i, j + len(floor_data)])  # Offset index for X_edge nodes

        # # Compute pairwise Euclidean distances between main data and edge
        # distances = euclidean_distances(floor_data[sensor_cols].values, edge_data[sensor_cols].values)
        # edge_index = []

        # # Construct edges using KNN based on Euclidean distance
        # for i in range(distances.shape[0]):
        #     # Get k nearest neighbors from X_edge for each node in the main data
        #     neighbors = np.argsort(distances[i])[:k]  # Take top k (smallest distances)
        #     for j in neighbors:
        #         edge_index.append([i, j + len(floor_data)])

        # Combine features from the floor data and X_edge for unified node features
        unified_features = np.vstack([floor_data[sensor_cols].values, edge_data[sensor_cols].values])
        unified_features = torch.tensor(unified_features, dtype=torch.float)

        # Convert edge list to PyTorch Geometric format
        edge_index = torch.tensor(edge_index, dtype=torch.long).T

        # Store the graph for the current floor
        graphs[floor_id] = Data(x=unified_features, edge_index=edge_index, y=y)

    return graphs

def create_weighted_knn_graphs_by_floor(data, sensor_cols, k, metric='cosine'):
    graphs = {}
    edge_copy = copy.deepcopy(df_edge)

    for floor_id in data['Floor_ID'].unique():
        # Filter data for the current floor
        floor_data = data[data['Floor_ID'] == floor_id]
        x = torch.tensor(floor_data[sensor_cols].values, dtype=torch.float)
        y = torch.tensor(floor_data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

        edge_data = edge_copy[edge_copy['Floor_ID'] == floor_id]

        if metric == 'cosine':
            # Compute pairwise cosine similarities between floor data and X_edge
            similarities = cosine_similarity(floor_data[sensor_cols].values, edge_data[sensor_cols].values)
            edge_index = []
            edge_weight = []

            # For each node in the current floor, find k most similar neighbors in X_edge
            for i in range(similarities.shape[0]):
                # Get indices of k most similar neighbors in X_edge
                neighbors = np.argsort(similarities[i])[-k:][::-1]  # Take top k
                for j in neighbors:
                    edge_index.append([i, j + len(floor_data)])  # Offset index for X_edge nodes
                    edge_weight.append(similarities[i, j])

        elif metric == 'euclidean':
            # Compute pairwise Euclidean distances between floor data and X_edge
            distances = euclidean_distances(floor_data[sensor_cols].values, edge_data[sensor_cols].values)
            edge_index = []
            edge_weight = []

            # For each node in the current floor, find k nearest neighbors in X_edge
            for i in range(distances.shape[0]):
                neighbors = np.argsort(distances[i])[:k]  # Take top k (smallest distances)
                for j in neighbors:
                    edge_index.append([i, j + len(floor_data)])
                    edge_weight.append(1 / (distances[i, j] + 1e-6))  # Use inverse distance as weight

        else:
            raise ValueError("Unsupported metric. Choose 'cosine' or 'euclidean'.")

        # Combine features from the floor data and X_edge for unified node features
        unified_features = np.vstack([floor_data[sensor_cols].values, edge_data[sensor_cols].values])
        unified_features = torch.tensor(unified_features, dtype=torch.float)

        # Convert edge list and weights to PyTorch Geometric format
        edge_index = torch.tensor(edge_index, dtype=torch.long).T
        edge_weight = torch.tensor(edge_weight, dtype=torch.float)

        # Store the graph for the current floor
        graphs[floor_id] = Data(x=unified_features, edge_index=edge_index, edge_attr=edge_weight, y=y)

    return graphs

graphs_by_floor = create_knn_graphs_by_floor(df_train, sensor_cols, k=5)
graphs_by_floor_val = create_knn_graphs_by_floor(df_val, sensor_cols, k=5)

In [ ]:
# class GraphSAGE(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels):
#         super(GraphSAGE, self).__init__()
#         self.conv1 = SAGEConv(in_channels, hidden_channels)
#         self.conv2 = SAGEConv(hidden_channels, out_channels)

#     def forward(self, x, edge_index):
#         x = self.conv1(x, edge_index).relu()
#         x = self.conv2(x, edge_index)
#         return x

from torch.nn import Dropout, LeakyReLU, BatchNorm1d, Linear
from torch.optim.lr_scheduler import StepLR
from torch_geometric.nn import GATConv, GraphConv, SAGEConv
import torch.nn.functional as F
# from torch_geometric.nn import BatchNorm

# class GAT(torch.nn.Module):
#     def __init__(self, in_channels, hidden_channels, out_channels, heads):
#         super(GAT, self).__init__()
#         self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, concat=True)
#         self.bn1 = BatchNorm1d(hidden_channels * heads)
#         self.activation = LeakyReLU(negative_slope=0.2)
#         self.dropout = Dropout(p=0.3)
#         self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=4, concat=False)

#     def forward(self, x, edge_index):
#         # x = self.conv1(x, edge_index)
#         # x = self.activation(x)
#         # x = self.dropout(x)
#         # x = self.bn1(x)
#         # x = self.conv2(x, edge_index)
#         x = self.conv1(x, edge_index)
#         x = self.activation(x)
#         x = self.dropout(x)
#         x = self.bn1(x)
#         x = self.conv2(x, edge_index)
#         return x

class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, heads):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, concat=True)
        self.activation = LeakyReLU(negative_slope=0.2)
        self.bn1 = BatchNorm1d(hidden_channels * heads)
        self.dropout = Dropout(p=0.3)
        self.conv2 = GATConv(hidden_channels * heads, hidden_channels, heads=4, concat=False)  # Single head in final layer
        # self.double()

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.bn1(x)
        x = self.conv2(x, edge_index)
        return x

# GraphSAGE Model
class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, aggr='mean'):
        super().__init__()
        # GraphSAGE layers
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr=aggr)
        self.activation = LeakyReLU(negative_slope=0.1)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.dropout = Dropout(p=0.4)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr=aggr)

    def forward(self, x, edge_index):
        # Apply first GraphSAGE layer
        x = self.conv1(x, edge_index)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.bn1(x)
        # Apply second GraphSAGE layer
        x = self.conv2(x, edge_index)
        return x

# GraphConv Model
class GraphConv1(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, aggr='mean'):
        super().__init__()
        # GraphConv layers
        self.conv1 = GraphConv(in_channels, hidden_channels, aggr=aggr)
        self.activation = LeakyReLU(negative_slope=0.2)
        self.bn1 = BatchNorm1d(hidden_channels)
        self.dropout = Dropout(p=0.3)
        self.conv2 = GraphConv(hidden_channels, hidden_channels, aggr=aggr)

    def forward(self, x, edge_index, edge_weight):
        # Apply first GraphConv layer with edge weights
        x = self.conv1(x, edge_index, edge_weight=edge_weight)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.bn1(x)
        # Apply second GraphConv layer with edge weights
        x = self.conv2(x, edge_index, edge_weight=edge_weight)
        return x



class MLP(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super().__init__()
        self.lin1 = Linear(hidden_channels, hidden_channels)
        self.activation = LeakyReLU(negative_slope=0.2)
        self.dropout = Dropout(p=0.5)
        self.lin2 = Linear(hidden_channels, out_channels)  # Output size is now dynamic
        # self.double()

    def forward(self, x):
        x = self.lin1(x)
        # x = F.relu(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.lin2(x)
        return x


class GATModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads):
        super().__init__()
        self.gat = GAT(in_channels, hidden_channels, heads)
        self.mlp = MLP(hidden_channels, out_channels)  # Match the output dimension of GAT
        # self.double()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.gat(x, edge_index)
        x = self.mlp(x)
        return x

class GraphSAGEModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, aggr='mean'):
        super().__init__()
        self.graphsage = GraphSAGE(in_channels, hidden_channels, aggr)
        self.mlp = MLP(hidden_channels, out_channels)
        # self.double()

    def forward(self, x, edge_index, edge_weight=None):
        x = self.graphsage(x, edge_index)
        x = self.mlp(x)
        return x

class GraphConvModel(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, aggr='mean'):
        super().__init__()
        self.graphconv = GraphConv1(in_channels, hidden_channels, aggr)
        self.mlp = MLP(hidden_channels, out_channels)
        # self.double()

    def forward(self, x, edge_index, edge_weight):
        x = self.graphconv(x, edge_index, edge_weight)
        x = self.mlp(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# # Initialize the GAT model
# model = GATModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     heads=4  # Multi-head attention for the first layer
# ).to(device)

# Initialize the GraphSAGE model
model = GraphSAGEModel(
    in_channels=len(sensor_cols),
    hidden_channels=128,
    out_channels=2,
    # out_channels=3,
    aggr='mean'
).to(device)

# # Initialize the GraphConv model
# model = GraphConvModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     # aggr='max'
# ).to(device)


In [ ]:
# # Initialize and train the model for one floor
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = GraphSAGE(in_channels=len(sensor_cols), hidden_channels=128, out_channels=2).to(device)

# # Train on a single floor and fine-tune on another
# def train_model(graph_data, model, epochs=100, lr=0.001):
#     graph_data = graph_data.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     criterion = torch.nn.MSELoss()

#     for epoch in range(epochs):
#         model.train()
#         optimizer.zero_grad()
#         pred = model(graph_data.x, graph_data.edge_index)
#         loss = criterion(pred, graph_data.y)
#         loss.backward()
#         optimizer.step()
#         if (epoch + 1) % 20 == 0:
#             print(f'Epoch {epoch+1}, Loss: {loss.item()}')
#     return model

# def train_model(graph_data, model, epochs=200, lr=0.001):
#     graph_data = graph_data.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     criterion = torch.nn.MSELoss()

#     for epoch in range(epochs):
#         model.train()
#         optimizer.zero_grad()
#         pred = model(graph_data.x, graph_data.edge_index)
#         loss = criterion(pred, graph_data.y)
#         loss.backward()
#         optimizer.step()
#         if (epoch + 1) % 20 == 0:
#             print(f'Epoch {epoch+1}, Loss: {loss.item()}')
#     return model

# def train_model(graph_data, model, epochs=500, lr=0.001, patience=40, min_delta=1e-4):
#     graph_data = graph_data.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=50, gamma=0.5)
#     criterion = torch.nn.MSELoss()

#     best_loss = float('inf')
#     patience_counter = 0

#     for epoch in range(epochs):
#         model.train()
#         optimizer.zero_grad()
#         pred = model(graph_data.x, graph_data.edge_index)
#         loss = criterion(pred, graph_data.y)
#         loss.backward()
#         optimizer.step()
#         # scheduler.step()

#         # # Early stopping check
#         # if loss.item() + min_delta < best_loss:
#         #     best_loss = loss.item()
#         #     patience_counter = 0
#         # else:
#         #     patience_counter += 1

#         if (epoch + 1) % 20 == 0:
#             print(f'Epoch {epoch+1}, Loss: {loss.item()}')

#         # if patience_counter >= patience:
#         #     print(f'Early stopping triggered at epoch {epoch+1}')
#         #     break

#     return model


def train_model(graph_data_train, model, graph_data_val, epochs=500, lr=0.001, patience=40, min_delta=1e-4):
    graph_data_train = graph_data_train.to(device)
    graph_data_val = graph_data_val.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
    criterion = torch.nn.MSELoss()
    # criterion = torch.nn.L1Loss()

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Training Phase
        model.train()
        optimizer.zero_grad()
        pred_train = model(graph_data_train.x, graph_data_train.edge_index)
        # train_loss = criterion(pred_train, graph_data_train.y)

        # Filter predictions to match labeled nodes
        labeled_output_train = pred_train[:graph_data_train.y.size(0)]

        # Compute loss only for labeled nodes
        train_loss = criterion(labeled_output_train, graph_data_train.y)
        train_loss.backward()
        optimizer.step()
        scheduler.step()

        # Validation Phase
        model.eval()
        with torch.no_grad():
            pred_val = model(graph_data_val.x, graph_data_val.edge_index)
            labeled_output_val = pred_val[:graph_data_val.y.size(0)]
            val_loss = criterion(labeled_output_val, graph_data_val.y)

        # Print progress every 20 epochs
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1}, Training Loss: {train_loss.item():.4f}, Validation Loss: {val_loss.item():.4f}")

    return model

def train_model_weight(graph_data_train, model, graph_data_val, epochs=500, lr=0.001, patience=40, min_delta=1e-4):
    graph_data_train = graph_data_train.to(device)
    graph_data_val = graph_data_val.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
    criterion = torch.nn.MSELoss()
    # criterion = torch.nn.L1Loss()

    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epochs):
        # Training Phase
        model.train()
        optimizer.zero_grad()
        pred_train = model(graph_data_train.x, graph_data_train.edge_index, graph_data_train.edge_weight)
        # train_loss = criterion(pred_train, graph_data_train.y)

        # Filter predictions to match labeled nodes
        labeled_output_train = pred_train[:graph_data_train.y.size(0)]

        # Compute loss only for labeled nodes
        train_loss = criterion(labeled_output_train, graph_data_train.y)
        train_loss.backward()
        optimizer.step()
        scheduler.step()

        # Validation Phase
        model.eval()
        with torch.no_grad():
            pred_val = model(graph_data_val.x, graph_data_val.edge_index, graph_data_val.edge_weight)
            labeled_output_val = pred_val[:graph_data_val.y.size(0)]
            val_loss = criterion(labeled_output_val, graph_data_val.y)

        # Print progress every 20 epochs
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch + 1}, Training Loss: {train_loss.item():.4f}, Validation Loss: {val_loss.item():.4f}")

    return model

In [ ]:
# from sklearn.metrics import r2_score

# # Evaluate the global model on a new dataset
# def evaluate_model(model, test_graph):
#     model.eval()
#     test_graph = test_graph.to(device)
#     with torch.no_grad():
#         pred = model(test_graph.x, test_graph.edge_index)
#         true = test_graph.y

#         # Calculate Mean Absolute Error (MAE)
#         mae = torch.mean(torch.abs(pred - true))

#         # Calculate Root Mean Square Error (RMSE)
#         rmse = torch.sqrt(torch.mean((pred - true) ** 2))

#         r2 = r2_score(true.cpu().numpy(), pred.cpu().numpy())

#     print("Evaluation Results:")
#     print(f"MAE: {mae.item():.4f}")
#     print(f"RMSE: {rmse.item():.4f}")
#     print(f"R2 Score: {r2:.4f}")
#     return mae.item(), rmse.item()


from sklearn.metrics import r2_score

# Evaluate the global model on a new dataset
def evaluate_model(model, test_graph):
    model.eval()
    test_graph = test_graph.to(device)
    with torch.no_grad():
        # pred = model(test_graph.x, test_graph.edge_index)  # Predict positions
        # predx = pred[:, :1]
        # predy = pred[:, 1:2]
        # true = test_graph.y  # True positions
        # truex = true[:, :1]
        # truey = true[:, 1:2]

        # Predict positions
        pred = model(test_graph.x, test_graph.edge_index)

        # Filter predictions to match labeled nodes
        pred = pred[:test_graph.y.size(0)]

        # Split predictions into x and y components
        predx = pred[:, :1]
        predy = pred[:, 1:2]

        # True positions
        true = test_graph.y
        truex = true[:, :1]
        truey = true[:, 1:2]

        position_error = torch.sqrt(((predx - truex) ** 2) + ((predy - truey) ** 2))

        # Calculate Mean Absolute Error (MAE)
        maex = torch.mean(torch.abs(predx - truex))
        medianaex = torch.median(torch.abs(predx - truex))

        # Calculate Root Mean Square Error (RMSE)
        rmsex = torch.sqrt(torch.mean((predx - truex) ** 2))

        # Calculate Mean Absolute Percentage Error (MAPE)
        mapex = torch.mean(torch.abs((predx - truex) / truex)) * 100

        # Calculate R2 Score
        r2x = r2_score(truex.cpu().numpy(), predx.cpu().numpy())

        # Calculate Mean Absolute Error (MAE)
        maey = torch.mean(torch.abs(predy - truey))

        medianaey = torch.median(torch.abs(predy - truey))

        # Calculate Root Mean Square Error (RMSE)
        rmsey = torch.sqrt(torch.mean((predy - truey) ** 2))

        # Calculate Mean Absolute Percentage Error (MAPE)
        mapey = torch.mean(torch.abs((predy - truey) / truey)) * 100

        # Calculate R2 Score
        r2y = r2_score(truey.cpu().numpy(), predy.cpu().numpy())

        # Mean and Median Position Errors
        mean_position_error = torch.mean(position_error)
        median_position_error = torch.median(position_error)

    print("Evaluation Results X:")
    print(f"MAE: {maex.item():.4f}") # Probably main evaluator
    print(f"MedianAE: {medianaex.item():.4f}") # test
    print(f"RMSE: {rmsex.item():.4f}")
    print(f"MAPE: {mapex.item():.4f}%")
    # print(f"R2 Score: {r2x:.4f}") # not effective because data not in a normal distribution

    print("Evaluation Results Y:")
    print(f"MAE: {maey.item():.4f}") # Probably main evaluator
    print(f"MedianAE: {medianaey.item():.4f}") # test
    print(f"RMSE: {rmsey.item():.4f}")
    print(f"MAPE: {mapey.item():.4f}%")

    print("Positioning Error:")
    print(f"Mean: {mean_position_error:.4f}")
    print(f"Median: {median_position_error:.4f}")

    # print(f"R2 Score: {r2y:.4f}") # not effective because data not in a normal distribution
    return maex.item(), maey.item(), medianaex.item(), medianaey.item(), median_position_error, mean_position_error

def evaluate_model_weighted(model, test_graph):
    model.eval()
    test_graph = test_graph.to(device)
    with torch.no_grad():
        # pred = model(test_graph.x, test_graph.edge_index)  # Predict positions
        # predx = pred[:, :1]
        # predy = pred[:, 1:2]
        # true = test_graph.y  # True positions
        # truex = true[:, :1]
        # truey = true[:, 1:2]

        # Predict positions
        pred = model(test_graph.x, test_graph.edge_index, test_graph.edge_weight)

        # Filter predictions to match labeled nodes
        pred = pred[:test_graph.y.size(0)]

        # Split predictions into x and y components
        predx = pred[:, :1]
        predy = pred[:, 1:2]

        # True positions
        true = test_graph.y
        truex = true[:, :1]
        truey = true[:, 1:2]

        position_error = torch.sqrt(((predx - truex) ** 2) + ((predy - truey) ** 2))

        # Calculate Mean Absolute Error (MAE)
        maex = torch.mean(torch.abs(predx - truex))
        medianaex = torch.median(torch.abs(predx - truex))

        # Calculate Root Mean Square Error (RMSE)
        rmsex = torch.sqrt(torch.mean((predx - truex) ** 2))

        # Calculate Mean Absolute Percentage Error (MAPE)
        mapex = torch.mean(torch.abs((predx - truex) / truex)) * 100

        # Calculate R2 Score
        r2x = r2_score(truex.cpu().numpy(), predx.cpu().numpy())

        # Calculate Mean Absolute Error (MAE)
        maey = torch.mean(torch.abs(predy - truey))

        medianaey = torch.median(torch.abs(predy - truey))

        # Calculate Root Mean Square Error (RMSE)
        rmsey = torch.sqrt(torch.mean((predy - truey) ** 2))

        # Calculate Mean Absolute Percentage Error (MAPE)
        mapey = torch.mean(torch.abs((predy - truey) / truey)) * 100

        # Calculate R2 Score
        r2y = r2_score(truey.cpu().numpy(), predy.cpu().numpy())

        # Mean and Median Position Errors
        mean_position_error = torch.mean(position_error)
        median_position_error = torch.median(position_error)

    print("Evaluation Results X:")
    print(f"MAE: {maex.item():.4f}") # Probably main evaluator
    print(f"MedianAE: {medianaex.item():.4f}") # test
    print(f"RMSE: {rmsex.item():.4f}")
    print(f"MAPE: {mapex.item():.4f}%")
    # print(f"R2 Score: {r2x:.4f}") # not effective because data not in a normal distribution

    print("Evaluation Results Y:")
    print(f"MAE: {maey.item():.4f}") # Probably main evaluator
    print(f"MedianAE: {medianaey.item():.4f}") # test
    print(f"RMSE: {rmsey.item():.4f}")
    print(f"MAPE: {mapey.item():.4f}%")

    print("Positioning Error:")
    print(f"Mean: {mean_position_error:.4f}")
    print(f"Median: {median_position_error:.4f}")

    # print(f"R2 Score: {r2y:.4f}") # not effective because data not in a normal distribution
    return maex.item(), maey.item(), medianaex.item(), medianaey.item(), median_position_error, mean_position_error

def create_knn_unified_graph(data, sensor_cols, k):
    # Features: RSSI values
    x = torch.tensor(data[sensor_cols].values, dtype=torch.float)
    # Labels: Coordinates (Pos_x, Pos_y)
    y = torch.tensor(data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

    # Compute pairwise cosine similarities between main data and edge
    similarities = cosine_similarity(data[sensor_cols].values, df_edge[sensor_cols].values)
    edge_index = []

    # Construct edges using KNN based on cosine similarity
    for i in range(similarities.shape[0]):
        # Get k most similar neighbors from X_edge for each node in the main data
        neighbors = np.argsort(similarities[i])[-k:][::-1]  # Take top k (highest value)
        for j in neighbors:
            edge_index.append([i, j + len(data)])  # Offset indices for X_edge nodes

    # # Compute pairwise Euclidean distances between main data and edge
    # distances = euclidean_distances(data[sensor_cols].values, df_edge[sensor_cols].values)
    # edge_index = []

    # # Construct edges using KNN based on Euclidean distance
    # for i in range(distances.shape[0]):
    #     # Get k nearest neighbors from X_edge for each node in the main data
    #     neighbors = np.argsort(distances[i])[:k]  # Take top k (smallest distances)
    #     for j in neighbors:
    #         edge_index.append([i, j + len(data)])

    # Combine features from main data and X_edge for unified node features
    unified_features = np.vstack([data[sensor_cols].values, df_edge[sensor_cols].values])
    unified_features = torch.tensor(unified_features, dtype=torch.float)

    # Convert edge list to PyTorch Geometric format
    edge_index = torch.tensor(edge_index, dtype=torch.long).T

    return Data(x=unified_features, edge_index=edge_index, y=y)

def create_weighted_knn_unified_graph(data, sensor_cols, k, metric='cosine'):
    # Features: RSSI values
    x = torch.tensor(data[sensor_cols].values, dtype=torch.float)
    # Labels: Coordinates (Pos_x, Pos_y)
    y = torch.tensor(data[['Pos_x', 'Pos_y']].values, dtype=torch.float)

    # Choose similarity/distance metric
    if metric == 'cosine':
        similarities = cosine_similarity(data[sensor_cols].values, df_edge[sensor_cols].values)
        edge_index = []
        edge_weight = []

        # Construct edges using KNN based on cosine similarity
        for i in range(similarities.shape[0]):
            neighbors = np.argsort(similarities[i])[-k:][::-1]  # Take top k (highest value)
            for j in neighbors:
                edge_index.append([i, j + len(data)])  # Offset indices for X_edge nodes
                edge_weight.append(similarities[i, j])

    elif metric == 'euclidean':
        distances = euclidean_distances(data[sensor_cols].values, df_edge[sensor_cols].values)
        edge_index = []
        edge_weight = []

        # Construct edges using KNN based on Euclidean distance
        for i in range(distances.shape[0]):
            neighbors = np.argsort(distances[i])[:k]  # Take top k (smallest distances)
            for j in neighbors:
                edge_index.append([i, j + len(data)])  # Offset indices for X_edge nodes
                edge_weight.append(1 / (distances[i, j] + 1e-6))  # Use inverse distance as weight

    else:
        raise ValueError("Unsupported metric. Choose 'cosine' or 'euclidean'.")

    # Combine features from main data and X_edge for unified node features
    unified_features = np.vstack([data[sensor_cols].values, df_edge[sensor_cols].values])
    unified_features = torch.tensor(unified_features, dtype=torch.float)

    # Convert edge list and weights to PyTorch Geometric format
    edge_index = torch.tensor(edge_index, dtype=torch.long).T
    edge_weight = torch.tensor(edge_weight, dtype=torch.float)

    return Data(x=unified_features, edge_index=edge_index, edge_attr=edge_weight, y=y)


In [ ]:
# def train_model_final(graph_data, model, epochs=500, lr=0.001):
#     graph_data = graph_data.to(device)
#     optimizer = torch.optim.Adam(model.parameters(), lr=lr)
#     criterion = torch.nn.MSELoss()

#     for epoch in range(epochs):
#         model.train()
#         optimizer.zero_grad()
#         pred = model(graph_data.x, graph_data.edge_index)
#         loss = criterion(pred, graph_data.y)
#         loss.backward()
#         optimizer.step()
#         if (epoch + 1) % 20 == 0:
#             print(f'Epoch {epoch+1}, Loss: {loss.item()}')
#     return model

def train_model_final(graph_data, model, val_graph, test_graph, epochs=500, lr=0.001, patience=40, min_delta=1e-4):
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    graph_data = graph_data.to(device)
    val_graph = val_graph.to(device)
    test_graph = test_graph.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
    criterion = torch.nn.MSELoss()
    # criterion = torch.nn.L1Loss()

    best_loss = float('inf')
    patience_counter = 0
    error_history = []

    # model_best = model
    # position_error_best = 100

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(graph_data.x, graph_data.edge_index)
        # loss = criterion(pred, graph_data.y)
        # Filter predictions to match labeled nodes

        labeled_output = pred[:graph_data.y.size(0)]

        # Compute loss only for labeled nodes
        loss = criterion(labeled_output, graph_data.y)
        loss.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(val_graph.x, val_graph.edge_index)

            # Compute validation loss for labeled nodes
            val_labeled_output = val_pred[:val_graph.y.size(0)]
            val_loss = criterion(val_labeled_output, val_graph.y)

            test_pred = model(test_graph.x, test_graph.edge_index)
            test_labeled_output = test_pred[:test_graph.y.size(0)]

            # Split predictions into x and y components
            predx = test_labeled_output[:, :1]
            predy = test_labeled_output[:, 1:2]

            # True positions
            true = test_graph.y
            truex = true[:, :1]
            truey = true[:, 1:2]

            position_error = torch.sqrt(((predx - truex) ** 2) + ((predy - truey) ** 2))
            mean_position_error = torch.mean(position_error)
            # if mean_position_error < position_error_best:
            #     position_error_best = mean_position_error
            #     model_best = model
            print(mean_position_error.item())
            error_history.append(mean_position_error.item())

        # Logging every 20 epochs
        if (epoch + 1) % 20 == 0:
            # current_lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}, Loss Val: {val_loss.item():.4f}')

    return model, error_history

def train_model_final_weighted(graph_data, model, val_graph, test_graph, epochs=500, lr=0.001, patience=40, min_delta=1e-4):
    # device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    graph_data = graph_data.to(device)
    val_graph = val_graph.to(device)
    test_graph = test_graph.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # optimizer = torch.optim.Adamax(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.5)
    criterion = torch.nn.MSELoss()
    # criterion = torch.nn.L1Loss()

    best_loss = float('inf')
    patience_counter = 0
    error_history = []

    # model_best = model
    # position_error_best = 100

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(graph_data.x, graph_data.edge_index, graph_data.edge_weight)
        # loss = criterion(pred, graph_data.y)
        # Filter predictions to match labeled nodes

        labeled_output = pred[:graph_data.y.size(0)]

        # Compute loss only for labeled nodes
        loss = criterion(labeled_output, graph_data.y)
        loss.backward()
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_pred = model(val_graph.x, val_graph.edge_index, val_graph.edge_weight)

            # Compute validation loss for labeled nodes
            val_labeled_output = val_pred[:val_graph.y.size(0)]
            val_loss = criterion(val_labeled_output, val_graph.y)

            test_pred = model(test_graph.x, test_graph.edge_index, test_graph.edge_weight)
            test_labeled_output = test_pred[:test_graph.y.size(0)]

            # Split predictions into x and y components
            predx = test_labeled_output[:, :1]
            predy = test_labeled_output[:, 1:2]

            # True positions
            true = test_graph.y
            truex = true[:, :1]
            truey = true[:, 1:2]

            position_error = torch.sqrt(((predx - truex) ** 2) + ((predy - truey) ** 2))
            mean_position_error = torch.mean(position_error)
            # if mean_position_error < position_error_best:
            #     position_error_best = mean_position_error
            #     model_best = model
            print(mean_position_error.item())
            error_history.append(mean_position_error.item())

        # Logging every 20 epochs
        if (epoch + 1) % 20 == 0:
            # current_lr = scheduler.get_last_lr()[0]
            print(f'Epoch {epoch+1}, Loss: {loss.item():.4f}, Loss Val: {val_loss.item():.4f}')

    return model, error_history

GNN (No Transfer Learning)

In [ ]:
device_nonTF = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# # Initialize the GAT model
# model_nonTF = GATModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     heads=4  # Multi-head attention for the first layer
# ).to(device_nonTF)

# Initialize the GraphSAGE model
model_nonTF = GraphSAGEModel(
    in_channels=len(sensor_cols),
    hidden_channels=128,
    out_channels=2,
    # out_channels=3,
    aggr='mean'
).to(device_nonTF)

# # Initialize the GraphConv model
# model_nonTF = GraphConvModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     # aggr='max'
# ).to(device_nonTF)

test_error_history = []

# all_train_graph = create_knn_unified_graph(df_train, sensor_cols, k=5)
# all_val_graph = create_knn_unified_graph(df_val, sensor_cols, k=5)
# all_test_graph = create_knn_unified_graph(df_test, sensor_cols, k=5)
# model_nonTF, test_error_history = train_model_final(all_train_graph, model_nonTF, all_val_graph, all_test_graph)

# Weighted
all_train_graph = create_weighted_knn_unified_graph(df_train, sensor_cols, k=5)
all_val_graph = create_weighted_knn_unified_graph(df_val, sensor_cols, k=5)
all_test_graph = create_weighted_knn_unified_graph(df_test, sensor_cols, k=5)
model_nonTF, test_error_history = train_model_final_weighted(all_train_graph, model_nonTF, all_val_graph, all_test_graph)

best_epoch = min(test_error_history)

# Evaluate the global model on the new graph
print("Evaluating on the new dataset...")
# evaluate_model(model_nonTF, all_test_graph)
evaluate_model_weighted(model_nonTF, all_test_graph)
print(f"Best Epoch: {best_epoch}")
print(model_nonTF)

63.42823791503906
62.94983673095703
63.00885772705078
63.183074951171875
63.297786712646484
63.02508544921875
62.3137321472168
61.73516082763672
61.146053314208984
60.712318420410156
60.137367248535156
59.29182434082031
58.59956741333008
57.93476104736328
57.046058654785156
56.085350036621094
55.08848571777344
54.169151306152344
53.2892951965332
52.1286506652832
Epoch 20, Loss: 1624.6866, Loss Val: 1492.3192
50.54641342163086
48.598472595214844
46.48856735229492
44.54690933227539
42.224449157714844
38.84845733642578
34.85388946533203
30.974720001220703
27.45777130126953
24.577335357666016
23.000320434570312
23.139310836791992
25.28620147705078
28.270689010620117
30.963804244995117
32.80120086669922
33.03330612182617
31.74726676940918
29.317129135131836
25.362367630004883
Epoch 40, Loss: 317.3882, Loss Val: 319.4980
20.42963218688965
15.92896556854248
14.872537612915039
16.772716522216797
20.078615188598633
22.620670318603516
20.9361572265625
15.501016616821289
12.480436325073242
12.881

GNN TF, but parent model is not TF (Schema A)

In [ ]:
# final_train_graph = create_knn_graphs_by_floor(df_train, sensor_cols, k=5)
# final_val_graph = create_knn_graphs_by_floor(df_val, sensor_cols, k=5)
# test_graph = create_knn_graphs_by_floor(df_test, sensor_cols, k=5)
# Weighted
final_train_graph = create_weighted_knn_graphs_by_floor(df_train, sensor_cols, k=5)
final_val_graph = create_weighted_knn_graphs_by_floor(df_val, sensor_cols, k=5)
test_graph = create_weighted_knn_graphs_by_floor(df_test, sensor_cols, k=5)
mae_x_total = 0
mae_y_total = 0
medianae_x_total = 0
medianae_y_total = 0
median_positioning_error_total = 0
mean_positioning_error_total = 0
curr_floor = 0
total_data_test = 0
test_error_history = []
test_error_total = 0

for floor_id in sorted(final_train_graph.keys()):
    test_floor_history = []
    print(f"Training/Fine-tuning on Floor {floor_id}...")
    # model_final_2, test_floor_history = train_model_final(final_train_graph[floor_id], model_nonTF, final_val_graph[floor_id], test_graph[floor_id])
    model_final_2, test_floor_history = train_model_final_weighted(final_train_graph[floor_id], model_nonTF, final_val_graph[floor_id], test_graph[floor_id])
    best_epoch = min(test_floor_history)
    print(f"Best Epoch: {best_epoch}")
    test_error_history.append(best_epoch)
    curr_floor = floor_id
    print("Evaluating model on floor " + str(floor_id) + "...")
    # mae_x, mae_y, medianae_x, medianae_y, median_positioning_error, mean_positioning_error = evaluate_model(model_final_2, test_graph[floor_id])
    mae_x, mae_y, medianae_x, medianae_y, median_positioning_error, mean_positioning_error = evaluate_model_weighted(model_final_2, test_graph[floor_id])
    filtered_data = df_test[df_test['Floor_ID'] == floor_id]

    # Get the number of rows
    num_rows = filtered_data.shape[0]
    total_data_test += num_rows
    mae_x_total += mae_x * num_rows
    medianae_x_total += medianae_x * num_rows
    mae_y_total += mae_y * num_rows
    medianae_y_total += medianae_y * num_rows
    median_positioning_error_total += median_positioning_error * num_rows
    mean_positioning_error_total += mean_positioning_error * num_rows
    test_error_total += best_epoch * num_rows

Streaming output truncated to the last 5000 lines.
8.085137367248535
8.200800895690918
8.24018669128418
8.213582038879395
8.129754066467285
8.01124382019043
7.912827968597412
7.810296058654785
7.699974060058594
7.6376824378967285
Epoch 400, Loss: 17.6795, Loss Val: 4.3087
7.598771095275879
7.587710380554199
7.61757230758667
7.605765342712402
7.6379475593566895
7.678765296936035
7.763802528381348
7.829575538635254
7.900852680206299
7.915920257568359
7.937148571014404
7.9334025382995605
7.970104217529297
8.012709617614746
8.052258491516113
8.068147659301758
8.067319869995117
8.040534019470215
8.001002311706543
7.979459285736084
Epoch 420, Loss: 18.4979, Loss Val: 3.6587
7.922228813171387
7.859631538391113
7.804139137268066
7.813179969787598
7.81207799911499
7.827271938323975
7.82197904586792
7.845191478729248
7.8768463134765625
7.888786315917969
7.8995842933654785
7.902089595794678
7.8756561279296875
7.824409008026123
7.802020072937012
7.819764614105225
7.836851596832275
7.83226251602172

In [ ]:
print("GNN Transfer Learning Schema B Result")

print("Evaluation Results X:")
print(f"MAE: {mae_x_total/total_data_test:.4f}")
print(f"MedianAE: {medianae_x_total/total_data_test:.4f}")

print("Evaluation Results Y:")
print(f"MAE: {mae_y_total/total_data_test:.4f}")
print(f"MedianAE: {medianae_y_total/total_data_test:.4f}")

print(f"Median Positioning Error: {median_positioning_error_total/total_data_test:.4f}")
print(f"Mean Positioning Error: {mean_positioning_error_total/total_data_test:.4f}")
print(f"Mean Positioning Error Best: {test_error_total/total_data_test:.4f}")

GAT Transfer Learning 2.0 Result
Evaluation Results X:
MAE: 7.2380
MedianAE: 6.1236
Evaluation Results Y:
MAE: 2.9133
MedianAE: 2.2706
Median Positioning Error: 7.1980
Mean Positioning Error: 8.2815
Mean Positioning Error Best: 6.7955


GNN Transfer Learning Schema A (TF)

In [ ]:
device_TF = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model_TF = GATModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     heads=5  # Multi-head attention for the first layer
# ).to(device_TF)

# Initialize the GraphSAGE model
model_TF = GraphSAGEModel(
    in_channels=len(sensor_cols),
    hidden_channels=128,
    out_channels=2,
    # out_channels=3,
    aggr='mean'
).to(device_TF)

# # Initialize the GraphConv model
# model_TF = GraphConvModel(
#     in_channels=len(sensor_cols),
#     hidden_channels=128,
#     out_channels=2,
#     # out_channels=3,
#     aggr='mean'
# ).to(device_TF)


train_graph = create_weighted_knn_graphs_by_floor(df_train, sensor_cols, k=5)
test_graph = create_weighted_knn_graphs_by_floor(df_test, sensor_cols, k=5)
val_graph = create_weighted_knn_graphs_by_floor(df_val, sensor_cols, k=5)
curr_floor = 0
total_data_test = 0
for floor_id in sorted(train_graph.keys()):
    print(f"Training/Fine-tuning on Floor {floor_id}...")
    model_TF = train_model_weight(train_graph[floor_id], model_TF, val_graph[floor_id])
    curr_floor = floor_id
    filtered_data = df_test[df_test['Floor_ID'] == floor_id]
    # Get the number of rows
    num_rows = filtered_data.shape[0]
    total_data_test += num_rows

In [ ]:
curr_floor = 0
total_data_test = 0
test_error_history = []
test_error_total = 0

for floor_id in sorted(train_graph.keys()):
    test_floor_history = []
    print(f"Training/Fine-tuning on Floor {floor_id}...")
    model_final_2, test_floor_history = train_model_final_weighted(train_graph[floor_id], model_TF, val_graph[floor_id], test_graph[floor_id])
    best_epoch = min(test_floor_history)
    print(f"Best Epoch: {best_epoch}")
    test_error_history.append(best_epoch)
    curr_floor = floor_id
    print("Evaluating model on floor " + str(floor_id) + "...")
    mae_x, mae_y, medianae_x, medianae_y, median_positioning_error, mean_positioning_error = evaluate_model_weighted(model_final_2, test_graph[floor_id])
    filtered_data = df_test[df_test['Floor_ID'] == floor_id]

    # Get the number of rows
    num_rows = filtered_data.shape[0]
    total_data_test += num_rows
    test_error_total += best_epoch * num_rows

print("GNN Transfer Learning Schema A Result")
print(f"Mean Positioning Error Best: {test_error_total/total_data_test:.4f}")